# TCGA Multimodal Data Exploration
Exploratory analysis of the `cache_data.zip` dataset provided by the supervisor.
Covers TCGA-LUAD and TCGA-LUSC cohorts with clinical, transcriptomics, WSI, and methylation modalities.

In [1]:
import pickle
import json
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/extracted/cache_data")
SPLITS_DIR = DATA_DIR / "splits"

## 1. Load raw data

In [2]:
with open(DATA_DIR / "tcga_luad_prepared_data.pkl", "rb") as f:
    luad_data = pickle.load(f)

with open(DATA_DIR / "tcga_lusc_prepared_data.pkl", "rb") as f:
    lusc_data = pickle.load(f)

print(f"LUAD patients: {len(luad_data)}")
print(f"LUSC patients: {len(lusc_data)}")

LUAD patients: 521
LUSC patients: 506


## 2. Inspect a single patient record

In [3]:
sample_id = list(luad_data.keys())[0]
sample = luad_data[sample_id]

print(f"Patient ID: {sample_id}")
print(f"Keys: {list(sample.keys())}")
print(f"clinical shape:       {sample['clinical'].shape}")
print(f"transcriptomics shape:{sample['transcriptomics'].shape}")
print(f"wsi shape:            {sample['wsi'].shape}")
print(f"methylation shape:    {sample['methylation'].shape}")
print(f"avail:                {sample['avail']}")
print(f"label:                {sample['label']}")

Patient ID: TCGA-05-4244
Keys: ['clinical', 'transcriptomics', 'wsi', 'methylation', 'avail', 'label']
clinical shape:       (63,)
transcriptomics shape:(1824,)
wsi shape:            (1024,)
methylation shape:    (16166,)
avail:                [1. 1. 1. 0.]
label:                {'OS6': nan, 'OS24': nan, 'OS': 0.0, 'OS.time': 0.0, 'DSS': 0.0, 'DSS.time': 0.0, 'DFI': nan, 'DFI.time': nan, 'PFI': 0.0, 'PFI.time': 0.0}


## 3. Load train/val/test splits

The supervisor's notebook uses:
- LUAD: `tcga_luad_DSS_k3_r1_test0.2_val0.2_seed42.json`
- LUSC: `tcga_lusc_DSS_k5_r1_test0.2_val0.2_seed42.json`

The split JSON structure is: `repeat_0 -> fold_0 -> {train, val, test}`

In [4]:
def load_split(split_file: str, repeat: int = 0, fold: int = 0):
    with open(SPLITS_DIR / split_file, "r") as f:
        splits = json.load(f)
    fold_data = list(splits[f"repeat_{repeat}"][fold].values())
    return fold_data[0], fold_data[1], fold_data[2]

luad_train, luad_val, luad_test = load_split("tcga_luad_DSS_k3_r1_test0.2_val0.2_seed42.json")
lusc_train, lusc_val, lusc_test = load_split("tcga_lusc_DSS_k5_r1_test0.2_val0.2_seed42.json")

print("LUAD split sizes:")
print(f"  train={len(luad_train)} | val={len(luad_val)} | test={len(luad_test)}")
print("LUSC split sizes:")
print(f"  train={len(lusc_train)} | val={len(lusc_val)} | test={len(lusc_test)}")

LUAD split sizes:
  train=253 | val=127 | test=95
LUSC split sizes:
  train=284 | val=72 | test=90


## 4. Modality availability analysis

The `avail` array encodes which modalities are present for each patient:
`[clinical, transcriptomics, wsi, methylation]`
A value of `1.0` means the modality is available, `0.0` means it is missing.
This is the key information for the orchestrator routing logic.

In [5]:
MODALITY_KEYS = ["clinical", "transcriptomics", "wsi", "methylation"]

def compute_modality_stats(patient_ids: list, raw_data: dict, split_name: str):
    records = [raw_data[pid] for pid in patient_ids if pid in raw_data]
    total = len(records)

    print(f"\n--- {split_name} (n={total}) ---")

    for i, modality in enumerate(MODALITY_KEYS):
        present = sum(1 for r in records if np.isclose(r["avail"][i], 1.0, rtol=1e-09, atol=1e-09))
        print(f"  {modality:<20} present: {present}/{total} ({present/total*100:.1f}%)")

    patterns = {}
    for r in records:
        pattern = tuple(int(v) for v in r["avail"])
        patterns[pattern] = patterns.get(pattern, 0) + 1

    print("\n  Modality patterns [clinical, transcriptomics, wsi, methylation]:")
    for pattern, count in sorted(patterns.items(), key=lambda x: -x[1]):
        pct = count / total * 100
        print(f"    {pattern} -> {count} patients ({pct:.1f}%)")

compute_modality_stats(luad_train, luad_data, "LUAD train")
compute_modality_stats(luad_val,   luad_data, "LUAD val")
compute_modality_stats(luad_test,  luad_data, "LUAD test")
compute_modality_stats(lusc_train, lusc_data, "LUSC train")


--- LUAD train (n=253) ---
  clinical             present: 253/253 (100.0%)
  transcriptomics      present: 252/253 (99.6%)
  wsi                  present: 228/253 (90.1%)
  methylation          present: 219/253 (86.6%)

  Modality patterns [clinical, transcriptomics, wsi, methylation]:
    (1, 1, 1, 1) -> 194 patients (76.7%)
    (1, 1, 1, 0) -> 33 patients (13.0%)
    (1, 1, 0, 1) -> 24 patients (9.5%)
    (1, 1, 0, 0) -> 1 patients (0.4%)
    (1, 0, 1, 1) -> 1 patients (0.4%)

--- LUAD val (n=127) ---
  clinical             present: 127/127 (100.0%)
  transcriptomics      present: 125/127 (98.4%)
  wsi                  present: 118/127 (92.9%)
  methylation          present: 117/127 (92.1%)

  Modality patterns [clinical, transcriptomics, wsi, methylation]:
    (1, 1, 1, 1) -> 106 patients (83.5%)
    (1, 1, 1, 0) -> 10 patients (7.9%)
    (1, 1, 0, 1) -> 9 patients (7.1%)
    (1, 0, 1, 1) -> 2 patients (1.6%)

--- LUAD test (n=95) ---
  clinical             present: 95/95 (100.0%)

## 5. Survival label inspection

Available label keys: OS, OS.time, DSS, DSS.time, DFI, DFI.time, PFI, PFI.time
We focus on DSS (Disease-Specific Survival) as used in the supervisor's splits.
A label value of `nan` means the outcome is not available for that patient.

In [6]:
def label_stats(patient_ids: list, raw_data: dict, target: str, split_name: str):
    records = [raw_data[pid] for pid in patient_ids if pid in raw_data]
    values = [r["label"][target] for r in records]
    valid = [v for v in values if not np.isnan(v)]
    events = sum(1 for v in valid if np.isclose(v, 1.0, rtol=1e-09, atol=1e-09))

    print(f"\n{split_name} | target={target}")
    print(f"  valid labels : {len(valid)}/{len(records)}")
    print(f"  events (=1)  : {events} ({events/len(valid)*100:.1f}%)")

label_stats(luad_train, luad_data, "DSS", "LUAD train")
label_stats(luad_val,   luad_data, "DSS", "LUAD val")
label_stats(luad_test,  luad_data, "DSS", "LUAD test")
label_stats(lusc_train, lusc_data, "DSS", "LUSC train")


LUAD train | target=DSS
  valid labels : 253/253
  events (=1)  : 60 (23.7%)

LUAD val | target=DSS
  valid labels : 127/127
  events (=1)  : 30 (23.6%)

LUAD test | target=DSS
  valid labels : 95/95
  events (=1)  : 23 (24.2%)

LUSC train | target=DSS
  valid labels : 284/284
  events (=1)  : 56 (19.7%)


## 6. Feature metadata

Column names are available for clinical, transcriptomics, and methylation.
WSI features come from a foundation model encoder and have no named columns.

In [7]:
with open(DATA_DIR / "tcga_luad_metadata.pkl", "rb") as f:
    luad_meta = pickle.load(f)

print(f"Metadata keys: {list(luad_meta.keys())}")

for key in luad_meta.keys():
    cols = luad_meta[key]
    if isinstance(cols, bool):
        print(f"\n  {key}: boolean = {cols}")
    else:
        print(f"\n  {key}: {len(cols)} features")
        print(f"  First 5: {list(cols)[:5]}")

Metadata keys: ['clinical_columns', 'transcriptomics_columns', 'methylation_columns', 'features_extracted']

  clinical_columns: 63 features
  First 5: ['cigarettes_per_day.exposures', 'years_smoked.exposures', 'pack_years_smoked.exposures', 'age_at_index.demographic', 'disease_type_Acinar Cell Neoplasms']

  transcriptomics_columns: 1824 features
  First 5: ['REACTOME_2_LTR_CIRCLE_FORMATION', 'REACTOME_ABACAVIR_TRANSMEMBRANE_TRANSPORT', 'REACTOME_ABC_FAMILY_PROTEINS_MEDIATED_TRANSPORT', 'REACTOME_ABC_TRANSPORTERS_IN_LIPID_HOMEOSTASIS', 'REACTOME_ABC_TRANSPORTER_DISORDERS']

  methylation_columns: 16166 features
  First 5: ['rs5931272', 'rs798149', 'rs5987737', 'rs1416770', 'rs6626309']

  features_extracted: boolean = True
